In [1]:
!pip install torch torchvision torchaudio --quiet
!pip install transformers peft accelerate bitsandbytes --quiet
!pip install openai-whisper --quiet
!pip install git+https://github.com/speechbrain/speechbrain.git@develop --quiet
!pip install deepface tf-keras opencv-python-headless --quiet
!pip install vaderSentiment langchain langchain-openai langchain-community --quiet
!pip install elevenlabs gradio python-dotenv scipy scikit-learn==1.6.1 soundfile --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 14.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━

In [2]:
!apt-get install -y ffmpeg --quiet

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [3]:
import os
os.makedirs("audio", exist_ok=True)
os.makedirs("frames", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("qlora_adapter_final", exist_ok=True)
print("Folders created!")

Folders created!


In [4]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

Mounted at /content/drive
Drive mounted!


In [5]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if any(file.endswith(ext) for ext in ['.pt', '.pkl', '.safetensors', '.mp4', '.zip']):
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath)
            print(f"{filepath} — {size/1024/1024:.1f} MB")

/content/drive/MyDrive/DATASCIENCE/UNIT 1/Unit 1-20220129.zip — 1.6 MB
/content/drive/MyDrive/MiniIsoLM/training_histories.pkl — 0.0 MB
/content/drive/MyDrive/MiniIsoLM/data/tokenizer.pkl — 0.0 MB
/content/drive/MyDrive/MiniIsoLM/checkpoints/Bigram_best.pt — 0.0 MB
/content/drive/MyDrive/MiniIsoLM/checkpoints/Transformer_best.pt — 4.6 MB
/content/drive/MyDrive/MiniIsoLM/checkpoints/Mamba_best.pt — 2.3 MB
/content/drive/MyDrive/dpo_youtube_titles/checkpoint-129/scaler.pt — 0.0 MB
/content/drive/MyDrive/dpo_youtube_titles/checkpoint-129/adapter_model.safetensors — 490.1 MB
/content/drive/MyDrive/dpo_youtube_titles/checkpoint-129/optimizer.pt — 249.5 MB
/content/drive/MyDrive/dpo_youtube_titles/checkpoint-129/scheduler.pt — 0.0 MB
/content/drive/MyDrive/dpo_youtube_titles/checkpoint-258/scaler.pt — 0.0 MB
/content/drive/MyDrive/dpo_youtube_titles/checkpoint-258/adapter_model.safetensors — 490.1 MB
/content/drive/MyDrive/dpo_youtube_titles/checkpoint-258/optimizer.pt — 249.5 MB
/content/dr

In [7]:
import os
import shutil

drive_path = "/content/drive/MyDrive/Truesignal"

for f in os.listdir(drive_path):
    src = os.path.join(drive_path, f)
    if os.path.isfile(src):
        shutil.copy(src, f"/content/{f}")
        print(f"Copied: {f}")

print("All files copied!")

Copied: preprocess.py
Copied: voice_agent.py
Copied: face_agent.py
Copied: text_agent.py
Copied: coach_agent.py
Copied: pipeline.py
Copied: mlp_checkpoint.zip
Copied: cidf_agent.py
Copied: app.py
Copied: interview.mp4
Copied: qlora_adapter_final.zip
All files copied!


In [8]:
import zipfile
import os

# Unzip QLoRA adapter
print("Unzipping QLoRA adapter...")
with zipfile.ZipFile("/content/qlora_adapter_final.zip", "r") as z:
    z.extractall("/content/qlora_adapter_final")
print("QLoRA unzipped!")

# Unzip MLP checkpoint
print("Unzipping MLP checkpoint...")
with zipfile.ZipFile("/content/mlp_checkpoint.zip", "r") as z:
    z.extractall("/content/models")
print("MLP unzipped!")

# Verify files
print("\nQLoRA files:")
for f in os.listdir("/content/qlora_adapter_final"):
    print(f"  {f}")

print("\nMLP files:")
for f in os.listdir("/content/models"):
    print(f"  {f}")

Unzipping QLoRA adapter...
QLoRA unzipped!
Unzipping MLP checkpoint...
MLP unzipped!

QLoRA files:
  merges.txt
  vocab.json
  video_preprocessor_config.json
  added_tokens.json
  tokenizer.json
  adapter_model.safetensors
  tokenizer_config.json
  preprocessor_config.json
  special_tokens_map.json
  chat_template.jinja
  README.md
  adapter_config.json

MLP files:
  scaler.pkl
  mlp_fusion.pt


In [9]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["ELEVENLABS_API_KEY"] = userdata.get("ELEVENLABS_API_KEY")

print("API keys loaded from Colab Secrets!")

API keys loaded from Colab Secrets!


In [10]:
import sys
sys.path.insert(0, '/content')

content = open('/content/voice_agent.py').read()
old = 'probs = out_prob[0].detach().numpy().tolist()'
new = 'probs = out_prob[0].detach().cpu().numpy().tolist()'
open('/content/voice_agent.py', 'w').write(content.replace(old, new))
print("Voice agent fixed!")

from preprocess import extract_audio, extract_frames, detect_pauses
print("Running preprocessing...")
extract_audio()
extract_frames()
pauses = detect_pauses()
print(f"Done! Found {len(pauses)} pauses.")

Voice agent fixed!
Running preprocessing...
Extracting audio from video...
Audio extracted successfully.
Extracting frames from video...
Frames extracted successfully.
Detecting pauses in audio...
Found 49 pauses. Saved to audio/pauses.json
Done! Found 49 pauses.


In [11]:
# Run voice agent
from voice_agent import run_voice_agent

print("Running Voice Agent...")
voice_results = run_voice_agent()
print(f"Done! Dominant vocal emotion: {voice_results['dominant_vocal_emotion']}")

Running Voice Agent...
Loading Whisper model...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 124MiB/s]


Transcribing audio...


INFO:speechbrain.utils.fetching:Fetch custom_interface.py: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached


Transcript: Hi, I'm really excited to be here today. I've been working on something I genuinely love. It started...
Loading SpeechBrain emotion model...


...

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached


label_encoder.txt:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: wav2vec2, model, label_encoder


Analyzing vocal emotion...


Dominant vocal emotion: sad
Voice results saved to audio/voice_results.json
Done! Dominant vocal emotion: sad


In [12]:
from face_agent import run_face_agent

print("Running Face Agent...")
face_results = run_face_agent()
print(f"Done! Dominant facial emotion: {face_results['dominant_facial_emotion']}")

26-05-07 17:10:41 - Directory /root/.deepface has been created
26-05-07 17:10:41 - Directory /root/.deepface/weights has been created
Running Face Agent...
Analyzing frames with DeepFace...
26-05-07 17:10:44 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 68.8MB/s]


Dominant facial emotion: neutral
Analyzing pause frame with GPT-4V: frames/frame_0001.jpg
Analyzing pause frame with GPT-4V: frames/frame_0003.jpg
Analyzing pause frame with GPT-4V: frames/frame_0006.jpg
Face results saved to audio/face_results.json
Done! Dominant facial emotion: neutral


In [13]:
from text_agent import run_text_agent

print("Running Text Agent...")
text_results = run_text_agent()
print(f"Done! Dominant text emotion: {text_results['dominant_text_emotion']}")

Running Text Agent...
Transcript loaded: Hi, I'm really excited to be here today. I've been working on something I genuinely love. It started...
Loading RoBERTa emotion model...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Classifying text emotions...
Dominant text emotion: joy
Running VADER sentiment analysis...
VADER scores: {'neg': 0.04, 'neu': 0.77, 'pos': 0.191, 'compound': 0.9952}
Text results saved to audio/text_results.json
Done! Dominant text emotion: joy


In [14]:
# Load QLORA model
import torch
from peft import PeftModel
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, BitsAndBytesConfig

print("Loading Qwen2-VL-7B base model (3-5 minutes)...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
base = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    quantization_config=bnb,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")
print("Base model loaded!")

print("Loading QLoRA adapter...")
model = PeftModel.from_pretrained(base, "/content/qlora_adapter_final/")
model.eval()
print("QLoRA model ready!")

Loading Qwen2-VL-7B base model (3-5 minutes)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Base model loaded!
Loading QLoRA adapter...
QLoRA model ready!


In [16]:
cidf_qlora_code = '''import json
import re
import numpy as np
import torch
from scipy.spatial.distance import cosine

# File paths
FACE_RESULTS_PATH = "audio/face_results.json"
VOICE_RESULTS_PATH = "audio/voice_results.json"
TEXT_RESULTS_PATH = "audio/text_results.json"
CIDF_RESULTS_PATH = "audio/cidf_results.json"

# MELD emotion categories
MELD_EMOTIONS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

# Label mappings for each modality
DF_TO_MELD = {
    "angry": "anger", "disgust": "disgust", "fear": "fear",
    "happy": "joy", "sad": "sadness", "surprise": "surprise", "neutral": "neutral"
}
SUPERB_TO_MELD = {
    "neu": "neutral", "hap": "joy", "ang": "anger", "sad": "sadness",
    "neutral": "neutral", "happy": "joy", "angry": "anger"
}
ROBERTA_TO_MELD = {
    "anger": "anger", "disgust": "disgust", "fear": "fear",
    "joy": "joy", "neutral": "neutral", "sadness": "sadness", "surprise": "surprise",
    "happy": "joy", "sad": "sadness", "angry": "anger"
}

def build_emotion_vector(emotion_probs, label_map):
    """Convert raw model probabilities to normalized 7d MELD emotion vector."""
    vec = np.zeros(7, dtype=np.float32)
    for raw_label, prob in emotion_probs.items():
        meld = label_map.get(raw_label.lower())
        if meld and meld in MELD_EMOTIONS:
            vec[MELD_EMOTIONS.index(meld)] += float(prob)
    if vec.sum() > 0:
        vec = vec / vec.sum()
    return vec

def compute_mas(vec_a, vec_b):
    """Compute Modal Agreement Score as cosine similarity."""
    if np.linalg.norm(vec_a) == 0 or np.linalg.norm(vec_b) == 0:
        return 0.0
    return float(1.0 - cosine(vec_a, vec_b))

def get_dominant_conflict(mas_fv, mas_ft, mas_vt):
    """Find modality pair with lowest agreement."""
    mas_scores = {
        "face_voice": round(float(mas_fv), 4),
        "face_text": round(float(mas_ft), 4),
        "voice_text": round(float(mas_vt), 4)
    }
    min_pair = min(mas_scores, key=mas_scores.get)
    conflict_map = {
        "face_voice": "Face and Voice are conflicting",
        "face_text": "Face and Text are conflicting",
        "voice_text": "Voice and Text are conflicting"
    }
    return mas_scores, min_pair, conflict_map[min_pair]

def parse_qlora_response(response, mas_fv, mas_ft, mas_vt):
    """
    Parse QLoRA model response to extract incongruence score.
    Handles both JSON format and plain key-value format.
    The model sometimes returns without curly braces so we handle both cases.
    """
    # Try 1: Standard JSON with curly braces
    try:
        json_match = re.search(r"\\{.*?\\}", response, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            score = float(result.get("incongruence_score", -1))
            if 0 <= score <= 1:
                print(f"Parsed from JSON: {score}")
                return score
    except:
        pass

    # Try 2: Plain key-value format without curly braces
    # e.g. "incongruence_score: 0.5901"
    try:
        score_match = re.search(r"incongruence_score[:\\s]+([0-9]*\\.?[0-9]+)", response)
        if score_match:
            score = float(score_match.group(1))
            if 0 <= score <= 1:
                print(f"Parsed from key-value: {score}")
                return score
    except:
        pass

    # Try 3: Any float number in response as last resort
    try:
        numbers = re.findall(r"0\\.[0-9]+", response)
        if numbers:
            score = float(numbers[0])
            if 0 <= score <= 1:
                print(f"Parsed first float found: {score}")
                return score
    except:
        pass

    # Fallback: Use MAS formula
    fallback = round(1.0 - float(np.mean([mas_fv, mas_ft, mas_vt])), 4)
    print(f"Using MAS fallback: {fallback}")
    return fallback

def run_qlora_inference(face_vec, voice_vec, text_vec, mas_fv, mas_ft, mas_vt, model, tokenizer):
    """Run QLoRA fine-tuned Qwen2-VL-7B inference."""
    prompt = f"""You are an emotional incongruence detection model.
Given the following multimodal emotion analysis:
- Face emotion vector: {dict(zip(MELD_EMOTIONS, face_vec.tolist()))}
- Voice emotion vector: {dict(zip(MELD_EMOTIONS, voice_vec.tolist()))}
- Text emotion vector: {dict(zip(MELD_EMOTIONS, text_vec.tolist()))}
- Modal Agreement Score (Face-Voice): {mas_fv:.4f}
- Modal Agreement Score (Face-Text): {mas_ft:.4f}
- Modal Agreement Score (Voice-Text): {mas_vt:.4f}

Is this utterance emotionally incongruent?
Answer with a JSON object containing:
- incongruent: true or false
- incongruence_score: float between 0 and 1
- dominant_conflict: the most conflicting modality pair
- confidence: float between 0 and 1
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"QLoRA raw response: {response}")

    score = parse_qlora_response(response, mas_fv, mas_ft, mas_vt)
    return round(score, 4)

def run_cidf_agent(model=None, tokenizer=None):
    """Main CIDF agent function."""
    with open(FACE_RESULTS_PATH, "r") as f:
        face_data = json.load(f)
    with open(VOICE_RESULTS_PATH, "r") as f:
        voice_data = json.load(f)
    with open(TEXT_RESULTS_PATH, "r") as f:
        text_data = json.load(f)

    face_vec = build_emotion_vector(face_data["face_emotion_vector"], DF_TO_MELD)
    voice_vec = build_emotion_vector(voice_data["voice_emotion_vector"], SUPERB_TO_MELD)
    text_vec = build_emotion_vector(text_data["text_emotion_vector"], ROBERTA_TO_MELD)

    mas_fv = compute_mas(face_vec, voice_vec)
    mas_ft = compute_mas(face_vec, text_vec)
    mas_vt = compute_mas(voice_vec, text_vec)

    mas_scores, dominant_conflict_pair, conflict_description = get_dominant_conflict(mas_fv, mas_ft, mas_vt)
    print(f"MAS Scores: {mas_scores}")

    if model is not None and tokenizer is not None:
        incongruence_score = run_qlora_inference(
            face_vec, voice_vec, text_vec,
            mas_fv, mas_ft, mas_vt,
            model, tokenizer
        )
        model_type = "qlora_cidf"
        print(f"Final QLoRA Incongruence Score: {incongruence_score}")
    else:
        incongruence_score = round(1.0 - float(np.mean([mas_fv, mas_ft, mas_vt])), 4)
        model_type = "mas_fallback"
        print(f"Fallback Score: {incongruence_score}")

    results = {
        "mas_scores": mas_scores,
        "incongruence_score": incongruence_score,
        "is_incongruent": incongruence_score > 0.5,
        "dominant_conflict_pair": dominant_conflict_pair,
        "conflict_description": conflict_description,
        "dominant_facial_emotion": face_data["dominant_facial_emotion"],
        "dominant_vocal_emotion": voice_data["dominant_vocal_emotion"],
        "dominant_text_emotion": text_data["dominant_text_emotion"],
        "model_type": model_type
    }

    with open(CIDF_RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)
    print(f"CIDF results saved.")
    return results
'''

with open("/content/cidf_agent_qlora.py", "w") as f:
    f.write(cidf_qlora_code)

import shutil
shutil.copy("/content/cidf_agent_qlora.py", "/content/drive/MyDrive/Truesignal/cidf_agent_qlora.py")
print("cidf_agent_qlora.py created and saved to Drive!")

cidf_agent_qlora.py created and saved to Drive!


In [17]:
# checking parser work
import importlib
import sys

# Clear any cached version
if 'cidf_agent_qlora' in sys.modules:
    del sys.modules['cidf_agent_qlora']

from cidf_agent_qlora import run_cidf_agent

print("Testing QLoRA CIDF agent...")
results = run_cidf_agent(model=model, tokenizer=tokenizer)
print(f"\nFinal Results:")
print(f"Incongruence Score: {results['incongruence_score']}")
print(f"Model Type: {results['model_type']}")
print(f"Dominant Conflict: {results['conflict_description']}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Testing QLoRA CIDF agent...
MAS Scores: {'face_voice': 0.0039, 'face_text': 0.2282, 'voice_text': 0.0085}
QLoRA raw response: incongruent: true
incongruence_score: 0.5400101010101011
dominant_conflict: ['face', 'voice']
confidence: 0.004080040040040041
Parsed from key-value: 0.540010101010101
Final QLoRA Incongruence Score: 0.54
CIDF results saved.

Final Results:
Incongruence Score: 0.54
Model Type: qlora_cidf
Dominant Conflict: Face and Voice are conflicting


In [19]:
app_qlora_code = '''import gradio as gr
import os
import shutil
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

from preprocess import extract_audio, extract_frames, detect_pauses
from voice_agent import run_voice_agent
from face_agent import run_face_agent
from text_agent import run_text_agent
from cidf_agent_qlora import run_cidf_agent
from coach_agent import run_coach_agent

openai_client = OpenAI()

EMOTION_EMOJI = {
    "happy": "😊", "sad": "😢", "angry": "😡",
    "neutral": "😐", "fear": "😨", "disgust": "🤢",
    "surprise": "😲", "joy": "😊", "sadness": "😢",
    "anger": "😡"
}

def get_emoji(emotion):
    return EMOTION_EMOJI.get(emotion.lower(), "😐")

def generate_general_suggestion(cidf_results):
    prompt = f"""
You are a communication coach. Give ONE short punchy tip (maximum 2 sentences) for someone whose:
- Face shows: {cidf_results["dominant_facial_emotion"]}
- Voice shows: {cidf_results["dominant_vocal_emotion"]}
- Words show: {cidf_results["dominant_text_emotion"]}
- Main conflict: {cidf_results["conflict_description"]}
Be direct, practical, energetic. No fluff. Start with a strong action verb.
"""
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=60
    )
    return response.choices[0].message.content.strip()

def run_truesignal(video_path):
    if video_path is None:
        yield ("Please upload a video first.", "", "", "", "", "", "", "", "", None)
        return
    try:
        shutil.copy(video_path, "interview.mp4")

        yield ("⚙️ Step 1/5: Preprocessing video...", "—", "—", "—", "—", "—", "—", "—", "—", None)
        extract_audio()
        extract_frames()
        detect_pauses()

        yield ("🎙️ Step 2/5: Analyzing voice...", "Processing...", "—", "—", "—", "—", "—", "—", "—", None)
        voice_results = run_voice_agent()
        v_emotion = voice_results["dominant_vocal_emotion"]
        v_emoji = get_emoji(v_emotion)

        yield ("👁️ Step 3/5: Analyzing face...", f"{v_emoji} {v_emotion.upper()}", "Processing...", "—", "—", "—", "—", "—", "—", None)
        face_results = run_face_agent()
        f_emotion = face_results["dominant_facial_emotion"]
        f_emoji = get_emoji(f_emotion)

        yield ("📋 Step 4/5: Analyzing text...", f"{v_emoji} {v_emotion.upper()}", f"{f_emoji} {f_emotion.upper()}", "Processing...", "—", "—", "—", "—", "—", None)
        text_results = run_text_agent()
        t_emotion = text_results["dominant_text_emotion"]
        t_emoji = get_emoji(t_emotion)

        yield ("🧠 Step 5/5: Computing incongruence with QLoRA CIDF...", f"{v_emoji} {v_emotion.upper()}", f"{f_emoji} {f_emotion.upper()}", f"{t_emoji} {t_emotion.upper()}", "Computing...", "—", "—", "—", "—", None)
        cidf_results = run_cidf_agent(model=model, tokenizer=tokenizer)

        mas = cidf_results["mas_scores"]
        score = cidf_results["incongruence_score"]
        conflict = cidf_results["conflict_description"]

        mas_display = f"🎙️ Face ↔ Voice:   {mas[\'face_voice\']:.4f}\\n👁️ Face ↔ Text:    {mas[\'face_text\']:.4f}\\n📋 Voice ↔ Text:  {mas[\'voice_text\']:.4f}"

        if score >= 0.7:
            score_label = f"🔴 {score:.4f} — HIGH INCONGRUENCE"
        elif score >= 0.5:
            score_label = f"🟡 {score:.4f} — MODERATE INCONGRUENCE"
        else:
            score_label = f"🟢 {score:.4f} — LOW INCONGRUENCE"

        yield ("🎤 Generating coaching...", f"{v_emoji} {v_emotion.upper()}", f"{f_emoji} {f_emotion.upper()}", f"{t_emoji} {t_emotion.upper()}", mas_display, score_label, conflict, "Generating...", "Generating...", None)

        general_suggestion = generate_general_suggestion(cidf_results)
        coaching_message = run_coach_agent()
        audio_path = "outputs/coaching_audio.mp3"

        yield ("✅ Analysis Complete!", f"{v_emoji} {v_emotion.upper()}", f"{f_emoji} {f_emotion.upper()}", f"{t_emoji} {t_emotion.upper()}", mas_display, score_label, conflict, general_suggestion, coaching_message, audio_path)

    except Exception as e:
        import traceback
        yield (f"❌ Error: {str(e)}\\n{traceback.format_exc()}", "—", "—", "—", "—", "—", "—", "—", "—", None)

css = """
@import url(\'https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&family=Space+Grotesk:wght@400;500;600;700&display=swap\');
body, .gradio-container { background: #eef2f7 !important; font-family: \'Inter\', sans-serif !important; color: #1a1a2e !important; }
.gradio-container { max-width: 1100px !important; margin: 0 auto !important; padding: 0 16px !important; }
#truesignal-header { background: linear-gradient(135deg, #0f172a 0%, #1e293b 60%, #0f3460 100%); border-radius: 16px; padding: 32px 40px; margin-bottom: 20px; text-align: center; position: relative; overflow: hidden; }
#truesignal-header::before { content: \'\'; position: absolute; top: 0; left: 0; right: 0; height: 2px; background: linear-gradient(90deg, transparent, #64b5f6, #ce93d8, #80cbc4, transparent); }
#ts-badge { background: rgba(100,181,246,0.12); border: 1px solid rgba(100,181,246,0.3); border-radius: 20px; padding: 5px 16px; font-size: 10px; color: #90caf9; letter-spacing: 2px; text-transform: uppercase; display: inline-block; margin-bottom: 16px; }
#ts-title { font-family: \'Space Grotesk\', sans-serif; font-size: 3.2em; font-weight: 700; letter-spacing: -2px; line-height: 1; margin: 0; }
#ts-title .t1 { color: #ffffff; } #ts-title .t2 { background: linear-gradient(135deg, #64b5f6, #ce93d8, #80cbc4); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; }
#ts-subtitle { font-size: 0.72em; color: #78909c; letter-spacing: 4px; text-transform: uppercase; margin-top: 12px; display: block; }
#ts-divider { width: 80px; height: 2px; background: linear-gradient(90deg, #64b5f6, #ce93d8); margin: 14px auto 0; border-radius: 2px; }
label span { font-family: \'Inter\', sans-serif !important; font-size: 0.72em !important; font-weight: 600 !important; letter-spacing: 2px !important; text-transform: uppercase !important; color: #78909c !important; }
textarea { background: #ffffff !important; border: 1px solid #e2e8f0 !important; border-radius: 10px !important; color: #1a1a2e !important; font-family: \'Inter\', sans-serif !important; font-size: 0.9em !important; padding: 12px 14px !important; box-shadow: 0 2px 8px rgba(0,0,0,0.04) !important; }
.upload-zone { border: 2px dashed #90caf9 !important; border-radius: 14px !important; background: #ffffff !important; box-shadow: 0 2px 8px rgba(0,0,0,0.05) !important; min-height: 180px !important; }
.analyze-btn { background: linear-gradient(135deg, #0f172a, #1e3a5f, #2d5986) !important; color: #ffffff !important; font-family: \'Inter\', sans-serif !important; font-weight: 700 !important; font-size: 1em !important; letter-spacing: 3px !important; text-transform: uppercase !important; border: none !important; border-radius: 10px !important; padding: 14px !important; width: 100% !important; box-shadow: 0 4px 15px rgba(30,58,95,0.3) !important; transition: all 0.3s ease !important; margin-top: 8px !important; }
.status-box textarea { color: #1e88e5 !important; text-align: center !important; font-weight: 600 !important; }
.tip-box textarea { color: #4e342e !important; font-size: 0.95em !important; line-height: 1.7 !important; border-left: 4px solid #f9a825 !important; background: #fffde7 !important; min-height: 100px !important; }
.voice-box textarea { color: #1e88e5 !important; font-weight: 700 !important; font-size: 1.1em !important; text-align: center !important; border-top: 3px solid #1e88e5 !important; }
.face-box textarea { color: #7b1fa2 !important; font-weight: 700 !important; font-size: 1.1em !important; text-align: center !important; border-top: 3px solid #7b1fa2 !important; }
.text-box textarea { color: #00897b !important; font-weight: 700 !important; font-size: 1.1em !important; text-align: center !important; border-top: 3px solid #00897b !important; }
.mas-box textarea { color: #1e88e5 !important; line-height: 2.2 !important; font-size: 0.88em !important; }
.score-box textarea { font-size: 1.2em !important; font-weight: 800 !important; text-align: center !important; color: #e53935 !important; }
.conflict-box textarea { color: #bf360c !important; font-weight: 600 !important; border-left: 4px solid #fb8c00 !important; }
.coaching-box textarea { color: #1a1a2e !important; font-size: 0.95em !important; line-height: 1.8 !important; border-left: 4px solid #00897b !important; background: #f0faf8 !important; }
.footer-area { text-align: center; padding: 20px 0 8px; color: #90a4ae; font-size: 0.75em; letter-spacing: 1px; border-top: 1px solid #e2e8f0; margin-top: 24px; font-family: \'Inter\', sans-serif; }
"""

with gr.Blocks(css=css, title="TrueSignal") as app:
    gr.HTML("""
    <div id="truesignal-header">
        <div id="ts-badge">⚡ Live Analysis System</div>
        <h1 id="ts-title"><span class="t1">True</span><span class="t2">Signal</span></h1>
        <span id="ts-subtitle">Real-Time Multimodal Emotional Incongruence Detection</span>
        <div id="ts-divider"></div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            video_input = gr.Video(label="📹 Upload Video", elem_classes=["upload-zone"])
            analyze_btn = gr.Button("⚡ Analyze", elem_classes=["analyze-btn"])
            status_output = gr.Textbox(label="Pipeline Status", interactive=False, elem_classes=["status-box"])
            suggestion_output = gr.Textbox(label="💡 Quick Tip", interactive=False, lines=4, elem_classes=["tip-box"])

        with gr.Column(scale=1):
            with gr.Row():
                voice_output = gr.Textbox(label="🎙️ Voice", interactive=False, elem_classes=["voice-box"])
                face_output = gr.Textbox(label="👁️ Face", interactive=False, elem_classes=["face-box"])
                text_output = gr.Textbox(label="📋 Text", interactive=False, elem_classes=["text-box"])
            mas_output = gr.Textbox(label="📊 Modal Agreement Scores", interactive=False, lines=3, max_lines=3, elem_classes=["mas-box"])
            with gr.Row():
                score_output = gr.Textbox(label="🔴 Incongruence Score", interactive=False, elem_classes=["score-box"])
                conflict_output = gr.Textbox(label="⚡ Dominant Conflict", interactive=False, elem_classes=["conflict-box"])

    with gr.Row():
        with gr.Column(scale=2):
            coaching_output = gr.Textbox(label="💬 Coaching Message", interactive=False, lines=4, elem_classes=["coaching-box"])
        with gr.Column(scale=1):
            audio_output = gr.Audio(label="🔊 Spoken Coaching", interactive=False)

    gr.HTML(\'<div class="footer-area">TrueSignal &nbsp;·&nbsp; STAT GR5293 &nbsp;·&nbsp; Columbia University &nbsp;·&nbsp; Spring 2026</div>\')

    analyze_btn.click(
        fn=run_truesignal,
        inputs=[video_input],
        outputs=[status_output, voice_output, face_output, text_output, mas_output, score_output, conflict_output, suggestion_output, coaching_output, audio_output]
    )

app.launch(share=True, debug=True)
'''

with open("/content/app_qlora.py", "w") as f:
    f.write(app_qlora_code)

import shutil
shutil.copy("/content/app_qlora.py", "/content/drive/MyDrive/Truesignal/app_qlora.py")
print("app_qlora.py created and saved to Drive!")

app_qlora.py created and saved to Drive!


In [25]:
import sys
if 'cidf_agent_qlora' in sys.modules:
    del sys.modules['cidf_agent_qlora']

import gradio as gr
gr.close_all()
import time
time.sleep(3)

exec(open("/content/app_qlora.py").read())

Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://001105b8f94ad75dd0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Extracting audio from video...
Audio extracted successfully.
Extracting frames from video...
Frames extracted successfully.
Detecting pauses in audio...
Found 49 pauses. Saved to audio/pauses.json
Loading Whisper model...
Transcribing audio...


INFO:speechbrain.utils.fetching:Fetch custom_interface.py: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached


Transcript: Hi, I'm really excited to be here today. I've been working on something I genuinely love. It started...
Loading SpeechBrain emotion model...


...

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:speechbrain.utils.fetching:Fetch wav2vec2.ckpt: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached
INFO:speechbrain.utils.fetching:Fetch model.ckpt: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain

Analyzing vocal emotion...
Dominant vocal emotion: sad
Voice results saved to audio/voice_results.json
Analyzing frames with DeepFace...
Dominant facial emotion: neutral
Analyzing pause frame with GPT-4V: frames/frame_0001.jpg
Analyzing pause frame with GPT-4V: frames/frame_0003.jpg
Analyzing pause frame with GPT-4V: frames/frame_0006.jpg
Face results saved to audio/face_results.json
Transcript loaded: Hi, I'm really excited to be here today. I've been working on something I genuinely love. It started...
Loading RoBERTa emotion model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Classifying text emotions...
Dominant text emotion: joy
Running VADER sentiment analysis...
VADER scores: {'neg': 0.04, 'neu': 0.77, 'pos': 0.191, 'compound': 0.9952}
Text results saved to audio/text_results.json
MAS Scores: {'face_voice': 0.0039, 'face_text': 0.2282, 'voice_text': 0.0085}
QLoRA raw response: incongruent: true
incongruence_score: 0.5907
dominant_conflict: ['face-voice']
confidence: 0.0040700000000000074
Parsed from key-value: 0.5907
Final QLoRA Incongruence Score: 0.5907
CIDF results saved.
Generating coaching message with GPT-4o...
Coaching message: It seems like there’s a disconnect between the neutral expression on your face and the sadness in your voice. To bring these into harmony, try practicing in front of a mirror while speaking, consciously matching your facial expressions with the emotions you're communicating through your voice. Remember, it's a journey to alignment, and you're making great progress with each step you take.
Converting coaching message to spe